# Eksplorasi & Pengecekan Data: Mortgages 2025 (`mortgages_2025_classified.csv`)

Notebook ini memuat data hipotek 2025 (**HMDA 2025**), memeriksa struktur data (dimensi, tipe data, memori), menganalisis missing values, serta menyediakan perbandingan indikator pinjaman antara **Bank Comparator vs Fintech**.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Konfigurasi path file data
DATA_PATH = Path("mortgages_2025_classified.csv")

# Memuat dataset (menangani penanda NA / Exempt pada data HMDA)
print("[*] Memuat data...")
df = pd.read_csv(
    DATA_PATH,
    low_memory=False,
    na_values=["NA", "Exempt", ""]
)
print(f"[✓] Data berhasil dimuat: {df.shape[0]:,} baris, {df.shape[1]} kolom.")

## 1. Struktur dan Dimensi Data
Memeriksa dimensi dataset, estimasi penggunaan memori, serta daftar seluruh kolom dan tipe datanya.

In [ ]:
# Estimasi memori dan dimensi
mem_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)
print(f"Total Baris    : {df.shape[0]:,}")
print(f"Total Kolom    : {df.shape[1]:,}")
print(f"Estimasi Memori: {mem_mb:.2f} MB")
print()

# Tabel ringkasan semua kolom
col_summary = pd.DataFrame({
    "No": range(1, len(df.columns) + 1),
    "Nama Kolom": df.columns,
    "Tipe Data": df.dtypes.values,
    "Jumlah Nilai Valid": df.notnull().sum().values,
    "Jumlah Missing": df.isnull().sum().values,
    "Persentase Missing (%)": ((df.isnull().sum() / len(df)) * 100).round(2).values
})
col_summary

## 2. Analisis Missing Values (Nilai Kosong)
Menampilkan kolom-kolom yang memiliki nilai kosong, diurutkan dari yang persentase missing-nya tertinggi ke terendah.

In [ ]:
missing_only = col_summary[col_summary["Jumlah Missing"] > 0].sort_values(
    by="Jumlah Missing", ascending=False
).reset_index(drop=True)

print(f"Total kolom dengan missing values: {len(missing_only)} dari {len(df.columns)} kolom.")
print()
missing_only

## 3. Kolom yang 100% Lengkap (0% Missing)
Daftar kolom yang tidak memiliki satupun nilai kosong.

In [ ]:
complete_cols = col_summary[col_summary["Jumlah Missing"] == 0]["Nama Kolom"].tolist()
print(f"Terdapat {len(complete_cols)} kolom lengkap tanpa nilai kosong:")
print()
for i, col in enumerate(complete_cols, 1):
    print(f"{i:2d}. {col}")

## 4. Preview Data (5 Baris Pertama)

In [ ]:
df.head()

## 5. Ringkasan Statistik Deskriptif Kolom Numerik Kunci

In [ ]:
key_cols = [
    "loan_amount", "interest_rate", "rate_spread", "total_loan_costs", 
    "combined_loan_to_value_ratio", "income", "loan_term"
]
df[key_cols].describe().T

---
## 6. Pengecekan Lanjutan (Custom Checks)
*Eksplorasi distribusi status pinjaman, lender teratas, dan kategori lender:*

In [ ]:
# Distribusi action_taken (status persetujuan pinjaman)
print("Distribusi action_taken:")
print(df["action_taken"].value_counts(dropna=False))

# Top 10 lender dengan pengajuan terbanyak
print()
print("Top 10 Lenders:")
print(df["lender_name"].value_counts().head(10))

In [ ]:
# Menghitung jumlah dari kategori lender type
print("Distribusi lender_type:")
print(df["lender_type"].value_counts(dropna=False))

---
## 7. Ringkasan Perbandingan Indikator: Bank Comparator vs Fintech
Analisis komparatif awal untuk indikator kunci:
- **Median & Rata-rata Rate Spread**
- **Median & Rata-rata Suku Bunga (Interest Rate)**
- **Median Plafon Pinjaman (Loan Amount)**
- **Median Biaya Origination (Origination Charges) & Rata-ratanya**
- **Rasio Biaya Origination terhadap Nilai Pinjaman (% Loan Amount)**
- **Median Total Seluruh Biaya Pinjaman (Total Loan Costs)**

In [ ]:
# ==============================================================================
# Ringkasan Perbandingan Indikator: Bank Comparator vs Fintech
# ==============================================================================

# 1. Pastikan kolom numerik relevan bertipe numerik
compare_metrics = ["rate_spread", "interest_rate", "origination_charges", "loan_amount", "total_loan_costs"]
for col in compare_metrics:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# 2. Hitung rasio origination charges terhadap nilai pinjaman (%)
df["orig_charge_pct"] = (df["origination_charges"] / df["loan_amount"]) * 100

# 3. Filter kelompok target (Bank Comparator vs Fintech)
target_groups = ["Bank comparator", "Fintech"]
subset = df[df["lender_type"].isin(target_groups)]

# 4. Agregasi statistik perbandingan
summary_table = pd.DataFrame({
    "Bank Comparator": [
        subset[subset["lender_type"] == "Bank comparator"]["loan_amount"].count(),
        subset[subset["lender_type"] == "Bank comparator"]["rate_spread"].median(),
        subset[subset["lender_type"] == "Bank comparator"]["rate_spread"].mean(),
        subset[subset["lender_type"] == "Bank comparator"]["interest_rate"].median(),
        subset[subset["lender_type"] == "Bank comparator"]["interest_rate"].mean(),
        subset[subset["lender_type"] == "Bank comparator"]["loan_amount"].median(),
        subset[subset["lender_type"] == "Bank comparator"]["loan_amount"].mean(),
        subset[subset["lender_type"] == "Bank comparator"]["origination_charges"].median(),
        subset[subset["lender_type"] == "Bank comparator"]["origination_charges"].mean(),
        subset[subset["lender_type"] == "Bank comparator"]["orig_charge_pct"].median(),
        subset[subset["lender_type"] == "Bank comparator"]["orig_charge_pct"].mean(),
        subset[subset["lender_type"] == "Bank comparator"]["total_loan_costs"].median(),
    ],
    "Fintech": [
        subset[subset["lender_type"] == "Fintech"]["loan_amount"].count(),
        subset[subset["lender_type"] == "Fintech"]["rate_spread"].median(),
        subset[subset["lender_type"] == "Fintech"]["rate_spread"].mean(),
        subset[subset["lender_type"] == "Fintech"]["interest_rate"].median(),
        subset[subset["lender_type"] == "Fintech"]["interest_rate"].mean(),
        subset[subset["lender_type"] == "Fintech"]["loan_amount"].median(),
        subset[subset["lender_type"] == "Fintech"]["loan_amount"].mean(),
        subset[subset["lender_type"] == "Fintech"]["origination_charges"].median(),
        subset[subset["lender_type"] == "Fintech"]["origination_charges"].mean(),
        subset[subset["lender_type"] == "Fintech"]["orig_charge_pct"].median(),
        subset[subset["lender_type"] == "Fintech"]["orig_charge_pct"].mean(),
        subset[subset["lender_type"] == "Fintech"]["total_loan_costs"].median(),
    ]
}, index=[
    "Jumlah Pinjaman (Volume)",
    "Median Rate Spread (%)",
    "Rata-rata Rate Spread (%)",
    "Median Suku Bunga (%)",
    "Rata-rata Suku Bunga (%)",
    "Median Nilai Pinjaman ($)",
    "Rata-rata Nilai Pinjaman ($)",
    "Median Biaya Origination ($)",
    "Rata-rata Biaya Origination ($)",
    "Median Rasio Biaya vs Pokok (%)",
    "Rata-rata Rasio Biaya vs Pokok (%)",
    "Median Total Biaya Pinjaman ($)"
])

# 5. Hitung selisih (Fintech - Bank Comparator)
summary_table["Selisih (Fintech - Bank)"] = summary_table["Fintech"] - summary_table["Bank Comparator"]

# 6. Format tampilan tabel yang rapi
display_table = summary_table.copy()
display_table.loc["Jumlah Pinjaman (Volume)"] = display_table.loc["Jumlah Pinjaman (Volume)"].apply(lambda x: f"{int(x):,}")
for idx in ["Median Rate Spread (%)", "Rata-rata Rate Spread (%)"]:
    display_table.loc[idx] = summary_table.loc[idx].apply(lambda x: f"{x:+.3f}%")
for idx in ["Median Suku Bunga (%)", "Rata-rata Suku Bunga (%)"]:
    display_table.loc[idx] = summary_table.loc[idx].apply(lambda x: f"{x:.3f}%" if pd.notnull(x) else "")
for idx in ["Median Nilai Pinjaman ($)", "Rata-rata Nilai Pinjaman ($)"]:
    display_table.loc[idx] = summary_table.loc[idx].apply(lambda x: f"${x:,.2f}")
for idx in ["Median Biaya Origination ($)", "Rata-rata Biaya Origination ($)", "Median Total Biaya Pinjaman ($)"]:
    display_table.loc[idx] = summary_table.loc[idx].apply(lambda x: f"${x:,.2f}")
for idx in ["Median Rasio Biaya vs Pokok (%)", "Rata-rata Rasio Biaya vs Pokok (%)"]:
    display_table.loc[idx] = summary_table.loc[idx].apply(lambda x: f"{x:.3f}%")

display_table